# TCC — Treino e avaliação no Colab (GPU)

Pipeline de análise de sobrevivência (SEER câncer de mama).

**Fluxo (rode as células em ordem):**
- Setup (células 1–6) → validar o CSV (célula 8).
- **Passo 2** — robustez dos modelos profundos em **VALIDAÇÃO** (não toca no teste).
- **Passo 3** — **AVALIAÇÃO FINAL**, que abre o **TESTE UMA ÚNICA VEZ**.

**Pré-requisitos:**
1. Código no GitHub: `CaetanoPorto/Deep-Learning-for-Survival-Analysis-Mortality-Risk-Prediction_2` (dê `git push` antes). Se o repo for **privado**, veja a nota na célula de clone.
2. CSV `breast_cancer.csv` no seu Google Drive (ex.: `MyDrive/TCC/breast_cancer.csv`). **O CSV nunca vai para o Git.**
3. Runtime com GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# O repo NÃO contém o CSV (gitignore).
# Repo PÚBLICO: a linha abaixo funciona direto.
# Repo PRIVADO: use um token -> https://<SEU_TOKEN>@github.com/CaetanoPorto/....git
!git clone https://github.com/CaetanoPorto/Deep-Learning-for-Survival-Analysis-Mortality-Risk-Prediction_2.git repo
%cd repo

In [ ]:
# Usa o torch GPU já instalado no Colab; instala só as libs de sobrevivência.
# (NÃO reinstalar torch/numpy/pandas para não derrubar o torch de GPU do Colab.)
!pip install -q scikit-survival lifelines pycox torchtuples

In [ ]:
import os
# >>> Ajuste para o caminho do SEU CSV no Drive: <<<
os.environ['SEER_CSV_PATH'] = '/content/drive/MyDrive/TCC/breast_cancer.csv'
assert os.path.exists(os.environ['SEER_CSV_PATH']), 'CSV não encontrado — ajuste o caminho acima'
print('CSV OK:', os.environ['SEER_CSV_PATH'])

## Passo 1 — validar que o CSV bate com o Perfil Empírico (~1 min)
Tem que dar **26/26 invariantes OK**. Se falhar, o CSV é diferente do registrado — pare.

In [ ]:
!python scripts/profile_dataset.py

## Passo 2 — robustez dos modelos profundos em VALIDAÇÃO (5 sementes)
Treina DeepSurv/DeepHit na base inteira e reporta média ± desvio entre sementes **no conjunto de validação** — o TESTE **não** é tocado aqui. Serve para ver se o C-index é estável entre sementes.

In [ ]:
!python scripts/run_deep.py --full --seeds 5

## Passo 3 — AVALIAÇÃO FINAL: abre o conjunto de TESTE UMA ÚNICA VEZ

São duas rodadas complementares (Protocolo de Validação + ADR-010):
- **3a — Tabela comparativa** dos 6 modelos (incl. GBS) na **amostra fixa** (100k), avaliada no teste dessa amostra. É a comparação principal (mesmo dado para todos).
- **3b — Sensibilidade na base inteira** com os modelos que escalam (Cox, Cox+splines, RSF, DeepSurv, DeepHit; **o GBS é pulado automaticamente** — ADR-010).

Cada rodada produz: tabela (Harrell/Uno/Antolini/IBS/Brier + IC bootstrap), calibração por decil, métricas por era, validação temporal 2010-15→2016-17 e sensibilidade do tempo desconhecido.

In [ ]:
# 3a — comparação principal na amostra fixa (o GBS aqui leva ~1-1,5 h; mantenha a aba ativa)
!python scripts/run_evaluation.py --sample-n 100000 --eval-set test --n-boot 200

In [ ]:
# 3b — sensibilidade na base inteira (sem GBS)
!python scripts/run_evaluation.py --full --eval-set test --n-boot 200